# Bookmarks and Favorites: Organize Your Resources

## Overview

Keep track of frequently used resources with the favorites feature. This notebook covers adding, listing, and removing favorites for easy access.

**What you'll learn:**
- Add mappings, snapshots, and instances to favorites
- List favorites by resource type
- Remove favorites
- Understand idempotency and cascade delete behavior

**Prerequisites:**
- Completed: [08_core_handling_errors.ipynb](./08_core_handling_errors.ipynb)
- Resources created to bookmark

**Time to complete:** 10 minutes

---

**Test Coverage:**
- Add favorites (mapping, snapshot, instance)
- List favorites (all and by resource type)
- Remove favorites
- Idempotency and cascade delete

In [ ]:
# Parameters cell - papermill will inject values here
# OPTIMIZATION: If SNAPSHOT_ID is provided, skip snapshot creation (~60s saved)
# The shared snapshot is created once at session start and reused by all tests
SNAPSHOT_ID = None  # Will be injected by papermill when running via pytest

## Setup

### Test Data Strategy

This test creates test resources (mapping, snapshot, instance) to favorite:
1. **Test Mapping**: Created for favoriting
2. **Test Snapshot**: Created for favoriting
3. **Test Instance**: Created for favoriting
4. **Favorites Testing**: Add, list, remove favorites
5. **Cleanup**: Automatic cleanup via atexit

In [ ]:
import os
import sys
import time

from graph_olap import notebook
from graph_olap.testing import TestPersona
from graph_olap.exceptions import ConflictError, NotFoundError, ValidationError
from graph_olap.models.mapping import EdgeDefinition, NodeDefinition, PropertyDefinition
from graph_olap_schemas import WrapperType

print(f"Python version: {sys.version}")
print(f"GRAPH_OLAP_API_URL: {os.environ.get('GRAPH_OLAP_API_URL', 'not set')}")

# Wake up Starburst Galaxy cluster (auto-suspends after 5 min idle)
notebook.wake_starburst()

In [ ]:
# Create test context as analyst Alice
ctx = notebook.test("FavTest", persona=TestPersona.ANALYST_ALICE)
client = ctx.client

print(f"Connected to {client._config.api_url}")
print(f"Test run ID: {ctx.run_id}")

## Create Test Resources

Create mapping, snapshot, and instance to test favorites API.

In [ ]:
# Define node and edge for test mapping
person_node = NodeDefinition(
    label="Person",
    sql="SELECT DISTINCT id, name, age FROM bigquery.graph_olap_e2e.persons",
    primary_key={"name": "id", "type": "STRING"},
    properties=[
        PropertyDefinition(name="name", type="STRING"),
        PropertyDefinition(name="age", type="INT64"),
    ]
)

knows_edge = EdgeDefinition(
    type="KNOWS",
    from_node="Person",
    to_node="Person",
    sql="SELECT DISTINCT from_id, to_id, since FROM bigquery.graph_olap_e2e.friendships",
    from_key="from_id",
    to_key="to_id",
    properties=[
        PropertyDefinition(name="since", type="INT64"),
    ]
)

# Create test mapping (auto-named, auto-tracked)
if SNAPSHOT_ID is not None:
    # Using shared snapshot - get mapping from snapshot
    print(f"Using shared SNAPSHOT_ID={SNAPSHOT_ID}, skipping mapping creation...")
    shared_snapshot = client.snapshots.get(SNAPSHOT_ID)
    TEST_MAPPING_ID = shared_snapshot.mapping_id
    TEST_MAPPING_NAME = f"shared-mapping-{TEST_MAPPING_ID}"
    print(f"Using shared mapping: id={TEST_MAPPING_ID}")
else:
    test_mapping = ctx.mapping(
        node_definitions=[person_node],
        edge_definitions=[knows_edge],
    )
    TEST_MAPPING_ID = test_mapping.id
    TEST_MAPPING_NAME = test_mapping.name
    print(f"Created test mapping: {TEST_MAPPING_NAME} (id={TEST_MAPPING_ID})")

In [ ]:
# Create test snapshot (auto-named, auto-tracked)
if SNAPSHOT_ID is not None:
    # Using shared snapshot - reuse it
    TEST_SNAPSHOT_ID = SNAPSHOT_ID
    TEST_SNAPSHOT_NAME = f"shared-snapshot-{TEST_SNAPSHOT_ID}"
    print(f"Using shared snapshot: id={TEST_SNAPSHOT_ID} (saved ~60s export time)")
else:
    print(f"Creating snapshot and waiting for export...")
    # Need to use client directly since we need custom params
    test_snapshot = ctx.snapshot(
        test_mapping,
        timeout=180,
    )
    TEST_SNAPSHOT_ID = test_snapshot.id
    TEST_SNAPSHOT_NAME = test_snapshot.name
    print(f"Created test snapshot: {TEST_SNAPSHOT_NAME} (id={TEST_SNAPSHOT_ID}, status={test_snapshot.status})")

In [ ]:
# Create test instance (auto-named, auto-tracked)
print(f"Creating instance and waiting for ready state...")

# Need to get the snapshot object for ctx.instance()
if SNAPSHOT_ID is not None:
    snapshot_obj = client.snapshots.get(SNAPSHOT_ID)
else:
    snapshot_obj = test_snapshot

test_instance = ctx.instance(
    snapshot_obj,
    wrapper_type=WrapperType.RYUGRAPH,
    timeout=180,
)
TEST_INSTANCE_ID = test_instance.id
TEST_INSTANCE_NAME = test_instance.name

print(f"Created test instance: {TEST_INSTANCE_NAME} (id={TEST_INSTANCE_ID}, status={test_instance.status})")

## 1. Favorites API Tests

### 1.1 Test Initial State - No Favorites

In [ ]:
# Test: Initially no favorites
all_favorites = client.favorites.list()
mapping_favorites = client.favorites.list(resource_type="mapping")
snapshot_favorites = client.favorites.list(resource_type="snapshot")
instance_favorites = client.favorites.list(resource_type="instance")

# Should have no favorites initially (clean state)
# Note: Other tests may have created favorites, so we just verify structure
assert all_favorites is not None, "Favorites list should not be None"
assert isinstance(all_favorites, list), "Favorites should be a list"
assert isinstance(mapping_favorites, list), "Mapping favorites should be a list"
assert isinstance(snapshot_favorites, list), "Snapshot favorites should be a list"
assert isinstance(instance_favorites, list), "Instance favorites should be a list"

print(f"FAV 1.1 PASSED: Initial favorites count: {len(all_favorites)}")
print(f"  Mappings: {len(mapping_favorites)}, Snapshots: {len(snapshot_favorites)}, Instances: {len(instance_favorites)}")

### 1.2 Test Add Mapping Favorite

In [ ]:
# Test: Add mapping to favorites
initial_count = len(client.favorites.list(resource_type="mapping"))

favorite = client.favorites.add(resource_type="mapping", resource_id=TEST_MAPPING_ID)

assert favorite is not None, "Favorite should not be None"
assert favorite.resource_type == "mapping", f"Expected resource_type 'mapping', got '{favorite.resource_type}'"
assert favorite.resource_id == TEST_MAPPING_ID, f"Expected resource_id {TEST_MAPPING_ID}, got {favorite.resource_id}"

# Verify it appears in list
mapping_favorites = client.favorites.list(resource_type="mapping")
assert len(mapping_favorites) == initial_count + 1, f"Expected {initial_count + 1} mapping favorites, got {len(mapping_favorites)}"

mapping_ids = [f.resource_id for f in mapping_favorites]
assert TEST_MAPPING_ID in mapping_ids, f"Test mapping {TEST_MAPPING_ID} should be in favorites"

print(f"FAV 1.2 PASSED: Added mapping {TEST_MAPPING_ID} to favorites")

### 1.3 Test Add Snapshot Favorite

In [ ]:
# Test: Add snapshot to favorites
initial_count = len(client.favorites.list(resource_type="snapshot"))

favorite = client.favorites.add(resource_type="snapshot", resource_id=TEST_SNAPSHOT_ID)

assert favorite is not None
assert favorite.resource_type == "snapshot"
assert favorite.resource_id == TEST_SNAPSHOT_ID

# Verify it appears in list
snapshot_favorites = client.favorites.list(resource_type="snapshot")
assert len(snapshot_favorites) == initial_count + 1

snapshot_ids = [f.resource_id for f in snapshot_favorites]
assert TEST_SNAPSHOT_ID in snapshot_ids

print(f"FAV 1.3 PASSED: Added snapshot {TEST_SNAPSHOT_ID} to favorites")

### 1.4 Test Add Instance Favorite

In [ ]:
# Test: Add instance to favorites
initial_count = len(client.favorites.list(resource_type="instance"))

favorite = client.favorites.add(resource_type="instance", resource_id=TEST_INSTANCE_ID)

assert favorite is not None
assert favorite.resource_type == "instance"
assert favorite.resource_id == TEST_INSTANCE_ID

# Verify it appears in list
instance_favorites = client.favorites.list(resource_type="instance")
assert len(instance_favorites) == initial_count + 1

instance_ids = [f.resource_id for f in instance_favorites]
assert TEST_INSTANCE_ID in instance_ids

print(f"FAV 1.4 PASSED: Added instance {TEST_INSTANCE_ID} to favorites")

### 1.5 Test List All Favorites

In [ ]:
# Test: List all favorites (no filter)
all_favorites = client.favorites.list()

assert all_favorites is not None
assert len(all_favorites) >= 3, f"Expected at least 3 favorites, got {len(all_favorites)}"

# Verify all our test resources are in the list (by resource_type + resource_id)
# Note: resource_id can collide across resource types (e.g., mapping 151 and snapshot 151)
# so we need to check using (resource_type, resource_id) as the compound key
favorites_by_key = {(f.resource_type, f.resource_id): f for f in all_favorites}

# Check each test resource is in favorites with correct type
mapping_key = ("mapping", TEST_MAPPING_ID)
snapshot_key = ("snapshot", TEST_SNAPSHOT_ID)
instance_key = ("instance", TEST_INSTANCE_ID)

assert mapping_key in favorites_by_key, f"Test mapping ({TEST_MAPPING_ID}) should be in all favorites. Keys: {[k for k in favorites_by_key.keys() if k[1] == TEST_MAPPING_ID]}"
assert snapshot_key in favorites_by_key, f"Test snapshot ({TEST_SNAPSHOT_ID}) should be in all favorites. Keys: {[k for k in favorites_by_key.keys() if k[1] == TEST_SNAPSHOT_ID]}"
assert instance_key in favorites_by_key, f"Test instance ({TEST_INSTANCE_ID}) should be in all favorites. Keys: {[k for k in favorites_by_key.keys() if k[1] == TEST_INSTANCE_ID]}"

# Verify the resource_type is correctly set on each favorite
assert favorites_by_key[mapping_key].resource_type == "mapping"
assert favorites_by_key[snapshot_key].resource_type == "snapshot"
assert favorites_by_key[instance_key].resource_type == "instance"

print(f"FAV 1.5 PASSED: Listed all favorites (found {len(all_favorites)} total)")
print(f"  All test resources present with correct types")

### 1.6 Test Add Duplicate (Should Raise ConflictError)

In [ ]:
# Test: Adding duplicate favorite returns ConflictError
initial_count = len(client.favorites.list(resource_type="mapping"))

# Add the same mapping again - should raise ConflictError
try:
    favorite = client.favorites.add(resource_type="mapping", resource_id=TEST_MAPPING_ID)
    raise AssertionError("Expected ConflictError when adding duplicate favorite, but call succeeded")
except ConflictError as e:
    # Expected behavior per API spec
    pass

# Count should not change
final_count = len(client.favorites.list(resource_type="mapping"))
assert final_count == initial_count, f"Duplicate add should not change count: {initial_count} -> {final_count}"

print(f"FAV 1.6 PASSED: Adding duplicate favorite correctly raises ConflictError (count stayed at {final_count})")

### 1.7 Test Remove Mapping Favorite

In [ ]:
# Test: Remove mapping favorite
initial_count = len(client.favorites.list(resource_type="mapping"))

client.favorites.remove(resource_type="mapping", resource_id=TEST_MAPPING_ID)

# Verify it's removed from list
mapping_favorites = client.favorites.list(resource_type="mapping")
assert len(mapping_favorites) == initial_count - 1, f"Expected {initial_count - 1} favorites, got {len(mapping_favorites)}"

mapping_ids = [f.resource_id for f in mapping_favorites]
assert TEST_MAPPING_ID not in mapping_ids, f"Mapping {TEST_MAPPING_ID} should not be in favorites after removal"

print(f"FAV 1.7 PASSED: Removed mapping {TEST_MAPPING_ID} from favorites")

### 1.8 Test Remove Snapshot Favorite

In [ ]:
# Test: Remove snapshot favorite
initial_count = len(client.favorites.list(resource_type="snapshot"))

client.favorites.remove(resource_type="snapshot", resource_id=TEST_SNAPSHOT_ID)

snapshot_favorites = client.favorites.list(resource_type="snapshot")
assert len(snapshot_favorites) == initial_count - 1

snapshot_ids = [f.resource_id for f in snapshot_favorites]
assert TEST_SNAPSHOT_ID not in snapshot_ids

print(f"FAV 1.8 PASSED: Removed snapshot {TEST_SNAPSHOT_ID} from favorites")

### 1.9 Test Remove Instance Favorite

In [ ]:
# Test: Remove instance favorite
initial_count = len(client.favorites.list(resource_type="instance"))

client.favorites.remove(resource_type="instance", resource_id=TEST_INSTANCE_ID)

instance_favorites = client.favorites.list(resource_type="instance")
assert len(instance_favorites) == initial_count - 1

instance_ids = [f.resource_id for f in instance_favorites]
assert TEST_INSTANCE_ID not in instance_ids

print(f"FAV 1.9 PASSED: Removed instance {TEST_INSTANCE_ID} from favorites")

### 1.10 Test Remove Non-Existent Favorite (Idempotency)

In [ ]:
# Test: Removing non-existent favorite is idempotent (should not error)
initial_count = len(client.favorites.list(resource_type="mapping"))

# Try to remove the already-removed mapping favorite again
client.favorites.remove(resource_type="mapping", resource_id=TEST_MAPPING_ID)

final_count = len(client.favorites.list(resource_type="mapping"))
assert final_count == initial_count, f"Remove should be idempotent: {initial_count} -> {final_count}"

print(f"FAV 1.10 PASSED: Removing non-existent favorite is idempotent")

### 1.11 Test Cascade Delete - Deleting Resource Deletes Favorites

In [ ]:
# Test: Cascade delete - deleting instance should delete its favorites
# Re-add instance to favorites (we removed it in test 1.9)
client.favorites.add(resource_type="instance", resource_id=TEST_INSTANCE_ID)

# Verify favorite exists
instance_favorites_before = client.favorites.list(resource_type="instance")
instance_ids_before = [f.resource_id for f in instance_favorites_before]
assert TEST_INSTANCE_ID in instance_ids_before, "Instance should be in favorites before delete"

# Terminate the instance (will be removed from ctx tracking)
print(f"Terminating instance {TEST_INSTANCE_ID} to test cascade delete...")
client.instances.terminate(TEST_INSTANCE_ID)

# Wait for instance to be fully terminated (async operation)
# Cascade delete of favorites happens when instance record is deleted from DB
max_wait = 60
poll_interval = 2
waited = 0
terminated = False

while waited < max_wait:
    try:
        instance = client.instances.get(TEST_INSTANCE_ID)
        if instance.status in ("terminated", "stopped"):
            print(f"Instance marked as terminated after {waited}s (status={instance.status})")
            # Continue waiting for full deletion
        else:
            print(f"Waiting for instance termination... (status={instance.status})")
    except NotFoundError:
        # Instance no longer exists - termination complete
        print(f"Instance fully deleted after {waited}s")
        terminated = True
        break
    time.sleep(poll_interval)
    waited += poll_interval

# Small grace period for cascade delete to propagate to favorites table
time.sleep(1)

# Verify favorite was CASCADE DELETED (not just marked as non-existent)
instance_favorites_after = client.favorites.list(resource_type="instance")
instance_ids_after = [f.resource_id for f in instance_favorites_after]
assert TEST_INSTANCE_ID not in instance_ids_after, "Instance favorite should be CASCADE DELETED when instance is terminated"

print(f"FAV 1.11 PASSED: Cascade delete works - favorite was deleted when instance was terminated")

## Summary

In [ ]:
# Cleanup is automatic via atexit - NotebookTest handles it
# Close client
client.close()

print("\n" + "="*60)
print("FAVORITES E2E TESTS COMPLETED!")
print("="*60)
print("\nValidated:")
print("  1. Favorites API:")
print("    1.1: Initial state verification")
print("    1.2: Add mapping favorite")
print("    1.3: Add snapshot favorite")
print("    1.4: Add instance favorite")
print("    1.5: List all favorites (with type filtering)")
print("    1.6: Add duplicate (idempotency)")
print("    1.7: Remove mapping favorite")
print("    1.8: Remove snapshot favorite")
print("    1.9: Remove instance favorite")
print("    1.10: Remove non-existent (idempotency)")
print("    1.11: Cascade delete (deleting resource deletes favorites)")
print("\nAll test resources cleaned up automatically via atexit")